# Generative AI as a Task Shock
## Full Empirical Pipeline Notebook

**Author:** Hakan Zeki Gülmez | **Supervisor:** Prof. Dr. Helmut Farbmacher | **TU Munich**

This notebook documents the complete empirical pipeline from raw data to final figures and tables.
All steps are reproducible. R-based DiD results are imported from `analysis/did_main.R`.

### Pipeline Overview
```
0. Setup & Data Loading
1. Sample Construction (funnel: 7509 → 94)
2. Literature Rubric Scoring (API + rubric)
3. Score Distribution & Validation  → Fig 1, Fig 2
4. DiD Results Summary              → Table 1, Table 2
5. Event Study & Placebo            → Fig 3
6. Revenue Divergence               → Fig 4
7. Firm-Level Evidence              → Table 5, Fig 8
8. Size Heterogeneity               → Table 3, Fig 5
9. Scoring Pipeline & Funnel        → Fig 6, Fig 7
10. Three-Period Analysis           → Table 6, Fig 9
11. Results Summary
```

### Key Results
| Spec | N | β | WCB p |
|------|---|---|-------|
| Lit Rubric SME $200M | 94 | -0.604 | 0.003*** |
| Lit Rubric SME $500M | 116 | -0.541 | 0.002*** |
| LLM Judge | 98 | -0.426 | 0.031** |
| O*NET alpha | 98 | -0.335 | 0.059* |
| Early AI era | 94 | -0.451 | 0.022** |
| Advanced AI era | 94 | -0.686 | 0.008*** |

## Section 0: Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from scipy import stats
import warnings, os
warnings.filterwarnings('ignore')

BASE = '..'

panel  = pd.read_csv(f'{BASE}/data/processed/master_panel.csv')
llm    = pd.read_csv(f'{BASE}/data/processed/llm_judge_scores.csv')
lit    = pd.read_csv(f'{BASE}/data/processed/lit_continuous_scores.csv')
onet   = pd.read_csv(f'{BASE}/data/processed/onet_similarity_scores.csv')
coefs  = pd.read_csv(f'{BASE}/data/processed/event_study_coefs.csv')

pre_rev = panel[panel['fiscal_year']<=2022].groupby(
    'ticker')['revenue'].median()/1e6
pre_rev = pre_rev.reset_index()
pre_rev.columns = ['ticker','pre_rev_m']

df = panel.merge(llm[['ticker','early_ai']], on='ticker')
df = df.merge(lit[['ticker','lit_score']], on='ticker', how='left')
df = df.merge(pre_rev, on='ticker')
df['year_quarter'] = df['fiscal_year'].astype(str)+'Q'+df['fiscal_quarter'].astype(str)

sme_lit = lit.merge(pre_rev, on='ticker')
sme_lit = sme_lit[sme_lit['pre_rev_m']<200]

print(f'Full panel firms:        {df["ticker"].nunique()}')
print(f'Literature rubric scored: {len(lit)}')
print(f'SME <$200M scored:       {len(sme_lit)}')
print(f'Period:                  {df["year_quarter"].min()} → {df["year_quarter"].max()}')
print(f'Lit score mean±SD:       {sme_lit["lit_score"].mean():.1f} ± {sme_lit["lit_score"].std():.1f}')

## Section 1: Sample Construction

Filter chain from NYSE/Nasdaq universe to primary analysis sample.

In [ ]:
print('=== SAMPLE CONSTRUCTION FUNNEL ===')
print(f'  NYSE/Nasdaq listed firms:            ~7,509')
print(f'  SIC 7370-7379 (software):              530')
print(f'  IPO before 2020Q1 + B2B:              173')
print(f'  Revenue data available (universe):    143')
print(f'  Literature Rubric scored (SME $500M): {len(lit.merge(pre_rev,on="ticker")[lambda x: x.pre_rev_m<500])}')
print(f'  Primary sample (SME $200M):           {len(sme_lit)}')
print()
print('Pre-shock revenue distribution (SME $200M):')
print(sme_lit['pre_rev_m'].describe().round(1))

## Section 2: Literature Rubric Scoring

Scoring logic: 10-K Item 1 → structured rubric → continuous score 1-100.

Rubric sources: Eloundou et al. (2024) E1/E0, Acemoglu & Restrepo (2022) automation criteria, Brynjolfsson et al. (2023) LLM suitability.

In [ ]:
SYSTEM_PROMPT = """You are an expert in AI economics applying a literature-grounded rubric.

Score B2B software firms on LLM replicability using:
ELOUNDOU ET AL. (2024): E1=text/knowledge/classification. E0=real-time data, physical, specialized tools.
ACEMOGLU & RESTREPO (2022): Automatable=codifiable routine cognitive. Non-automatable=complex judgment.
BRYNJOLFSSON ET AL. (2023): LLM-suitable=written communication, information synthesis, document processing.

SCALE:
1-20:  Pure E0 (infrastructure, security, hardware)
21-40: Mostly E0 (deep integration, proprietary data)
41-60: Mixed
61-80: Mostly E1 (text, documents, classification)
81-100: Pure E1 (core entirely text/knowledge)

Return ONLY: {\"score\": 45, \"reasoning\": \"brief reason\"}"""

print('=== SCORING PROCEDURE ===')
print('Input:  10-K Item 1 Business Description (~3000 chars, pre-shock)')
print('Model:  claude-opus-4-5')
print('Output: Continuous score 1-100 + reasoning')
print()
print('Example scores (SME $200M):')
examples = sme_lit.nlargest(5,'lit_score')[['ticker','lit_score','reasoning']]
examples = pd.concat([examples, sme_lit.nsmallest(5,'lit_score')[['ticker','lit_score','reasoning']]])
for _, r in examples.iterrows():
    print(f'  {r["ticker"]:6s} score={r["lit_score"]:3.0f}  {str(r["reasoning"])[:60]}')

## Section 3: Score Distribution & Construct Validity → Fig 1, Fig 2

In [ ]:
# Figure 1: Score distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 1: Literature-Grounded Replicability Score Distribution\n'
    'SME Sample (<$200M, n=94)', fontsize=11, fontweight='bold')

ax = axes[0]
ax.hist(sme_lit['lit_score'], bins=range(10,90,5),
    color='#7f8c8d', edgecolor='white', alpha=0.85)
for v, c, lbl in [
    (sme_lit['lit_score'].quantile(0.25), '#2980b9', f"Q25={sme_lit['lit_score'].quantile(0.25):.0f}"),
    (sme_lit['lit_score'].quantile(0.75), '#e74c3c', f"Q75={sme_lit['lit_score'].quantile(0.75):.0f}"),
    (sme_lit['lit_score'].mean(), 'black', f"Mean={sme_lit['lit_score'].mean():.1f}"),
]:
    ax.axvline(v, color=c, ls='--', lw=2, label=lbl)
ax.set_xlabel('Literature Replicability Score (1-100)', fontsize=10)
ax.set_ylabel('Number of Firms', fontsize=10)
ax.set_title('Panel A: Score Distribution', fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
bench = pd.concat([
    sme_lit.nlargest(8,'lit_score')[['ticker','lit_score']],
    sme_lit.nsmallest(8,'lit_score')[['ticker','lit_score']]
]).sort_values('lit_score')
med = sme_lit['lit_score'].median()
colors = ['#e74c3c' if v>med else '#2980b9' for v in bench['lit_score']]
bars = ax.barh(bench['ticker'], bench['lit_score'], color=colors, alpha=0.85)
for bar, val in zip(bars, bench['lit_score']):
    ax.text(val+0.5, bar.get_y()+bar.get_height()/2, f'{val:.0f}', va='center', fontsize=8)
ax.axvline(med, color='black', ls='--', lw=1.5, label=f'Median={med:.0f}')
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Above median'),
    mpatches.Patch(color='#2980b9', label='Below median')
], fontsize=8)
ax.set_title('Panel B: Highest and Lowest Scored Firms', fontsize=10)
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'{BASE}/figures/fig1_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig1_score_distribution.png')

In [ ]:
# Figure 2: Construct validity
merged = sme_lit.merge(onet[['ticker','firm_alpha']], on='ticker')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 2: Construct Validity — Three Replicability Measures\nSME <$200M',
    fontsize=11, fontweight='bold')

for ax, xcol, xlabel, title in [
    (axes[0], 'early_ai', 'LLM Judge Score', 'Panel A: LLM Judge vs Lit Rubric (r=0.895)'),
    (axes[1], 'firm_alpha', 'O*NET firm_alpha', 'Panel B: O*NET vs Lit Rubric (r=0.299)')
]:
    valid = merged[[xcol,'lit_score']].dropna()
    r, _ = stats.pearsonr(valid[xcol], valid['lit_score'])
    ax.scatter(valid[xcol], valid['lit_score'], alpha=0.6, s=40,
        color='#2980b9', edgecolors='white', linewidths=0.3)
    z = np.polyfit(valid[xcol], valid['lit_score'], 1)
    x_line = np.linspace(valid[xcol].min(), valid[xcol].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'r--', lw=1.5, alpha=0.7)
    ax.text(0.05, 0.95, f'r = {r:.3f}', transform=ax.transAxes,
        va='top', fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Literature Rubric Score', fontsize=10)
    ax.set_title(title, fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/figures/fig2_construct_validity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig2_construct_validity.png')
print(f'LLM Judge r: {merged["early_ai"].corr(merged["lit_score"]):.3f}')
print(f'O*NET r:     {merged["firm_alpha"].corr(merged["lit_score"]):.3f}')

## Section 4: DiD Results Summary → Table 1, Table 2

Regression run in R (`analysis/did_main.R`). Results imported here.

In [ ]:
print('=== TABLE 1: PRIMARY RESULTS ===')
t1 = pd.DataFrame({
    'Sample':  ['SME <$200M', 'SME <$500M'],
    'N':       [94, 116],
    'Beta':    [-0.604, -0.541],
    'SE':      [0.217, 0.202],
    'WCB_p':   [0.003, 0.002],
    'Sig':     ['***', '***']
})
print(t1.to_string(index=False))

print('\n=== TABLE 2: ROBUSTNESS ===')
t2 = pd.DataFrame({
    'Method':  ['LLM Judge (holistic)', 'O*NET alpha (task-based)'],
    'N':       [98, 98],
    'Beta':    [-0.426, -0.335],
    'WCB_p':   [0.031, 0.059],
    'Sig':     ['**', '*'],
    'r_lit':   [0.895, 0.299]
})
print(t2.to_string(index=False))

print('\nIdentification:')
print('  Placebo p:       0.157 ✅')
print('  Pre-trends:      0/11 sig ✅')
print('  GM null:         p>0.4 ✅')
print('  Firm scatter:    r=-0.274, p=0.008 ✅')

## Section 5: Event Study & Placebo → Fig 3

In [ ]:
coefs_s = coefs.sort_values('quarter').reset_index(drop=True)
quarters = coefs_s['quarter'].tolist()
shock_x = quarters.index('2022Q4')
x = list(range(len(quarters)))
pre_x  = [i for i,q in enumerate(quarters) if q<='2022Q3']
post_x = [i for i,q in enumerate(quarters) if q>'2022Q3']

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(x, coefs_s['ci_lo'], coefs_s['ci_hi'],
    alpha=0.15, color='#e74c3c', label='95% CI')
ax.axhline(y=0, color='black', lw=0.8)
ax.axvline(x=shock_x, color='black', ls='--', lw=1.5,
    alpha=0.8, label='ChatGPT (2022Q4)')
ax.plot(pre_x,  coefs_s['beta'].iloc[pre_x],
    'o-', color='#2980b9', lw=2, ms=5, label='Pre-shock β')
ax.plot(post_x, coefs_s['beta'].iloc[post_x],
    's-', color='#e74c3c', lw=2, ms=5, label='Post-shock β')

pre_avg  = coefs_s[coefs_s['quarter']<='2022Q3']['beta'].mean()
post_avg = coefs_s[coefs_s['quarter']>'2022Q3']['beta'].mean()
ax.set_ylabel('Differential Effect on ln(Revenue)', fontsize=10)
ax.set_title(f'Figure 3: Event Study — Parallel Pre-Trends Test\n'
    f'Pre-shock avg β={pre_avg:+.3f} (0/11 sig) | '
    f'Post-shock avg β={post_avg:+.3f}',
    fontsize=10, fontweight='bold')
ax.set_xticks(x[::2])
ax.set_xticklabels(quarters[::2], rotation=45, ha='right', fontsize=7)
ax.legend(fontsize=8, loc='lower left'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/figures/fig3_event_study.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3_event_study.png')
print(f'Pre-shock quarters significant: 0/11 ✅')
print(f'Placebo p = 0.157 ✅')

## Section 6: Revenue Divergence → Fig 4

In [ ]:
sme_q = df[(df['pre_rev_m']<200) & df['lit_score'].notna()].copy()
med_lit = sme_q[~sme_q['ticker'].duplicated()]['lit_score'].median()
sme_q['group'] = np.where(sme_q['lit_score']>=med_lit, 'HIGH', 'LOW')

base = sme_q[sme_q['fiscal_year']==2021].groupby('ticker')['revenue'].mean().reset_index()
base.columns = ['ticker','base_rev']
sme_q = sme_q.merge(base, on='ticker')
sme_q['rev_idx'] = sme_q['revenue']/sme_q['base_rev']*100

qtr = sme_q[sme_q['fiscal_year'].between(2020,2025)].groupby(
    ['year_quarter','group']).agg(
    med_idx=('rev_idx','median')).reset_index()

high = qtr[qtr['group']=='HIGH'].sort_values('year_quarter')
low  = qtr[qtr['group']=='LOW'].sort_values('year_quarter')
comp = high.merge(low, on='year_quarter', suffixes=('_h','_l'))
comp['gap'] = comp['med_idx_h'] - comp['med_idx_l']

quarters2 = comp['year_quarter'].tolist()
shock_x2  = quarters2.index('2022Q4') + 0.5
x2 = list(range(len(comp)))
n_h = sme_q[sme_q['group']=='HIGH']['ticker'].nunique()
n_l = sme_q[sme_q['group']=='LOW']['ticker'].nunique()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle(f'Figure 4: Revenue Divergence — High vs Low Replicability\n'
    f'SME <$200M | HIGH n={n_h}, LOW n={n_l}',
    fontsize=11, fontweight='bold')

ax = axes[0]
ax.plot(x2, comp['med_idx_h'], 'o-', color='#e74c3c', lw=2, ms=4,
    label=f'High Replicability (n={n_h})')
ax.plot(x2, comp['med_idx_l'], 's-', color='#2980b9', lw=2, ms=4,
    label=f'Low Replicability (n={n_l})')
ax.axvline(x=shock_x2, color='black', ls='--', lw=1.5, label='ChatGPT (2022Q4)')
ax.set_ylabel('Revenue Index (2021=100)', fontsize=9)
ax.set_title('Panel A: Revenue Index Trajectories', fontsize=10)
ax.set_xticks(x2[::2])
ax.set_xticklabels(quarters2[::2], rotation=45, ha='right', fontsize=7)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
pr = float(np.mean(comp['gap'][list(q<'2022Q4' for q in quarters2)]))
po = float(np.mean(comp['gap'][list(q>='2022Q4' for q in quarters2)]))
colors_bar = ['#e74c3c' if g<0 else '#27ae60' for g in comp['gap']]
ax.bar(x2, comp['gap'], color=colors_bar, alpha=0.75, width=0.7)
ax.axvline(x=shock_x2, color='black', ls='--', lw=1.5)
ax.axhline(y=0, color='black', lw=0.8)
ax.axhline(y=pr, color='gray', ls=':', lw=2, label=f'Pre avg: {pr:+.1f}')
ax.axhline(y=po, color='#e74c3c', ls=':', lw=2, label=f'Post avg: {po:+.1f}')
ax.set_ylabel('Revenue Gap HIGH−LOW (pts)', fontsize=9)
ax.set_title(f'Panel B: Revenue Gap | DiD = {po-pr:+.1f} pts', fontsize=10)
ax.set_xticks(x2[::2])
ax.set_xticklabels(quarters2[::2], rotation=45, ha='right', fontsize=7)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/figures/fig4_revenue_divergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig4_revenue_divergence.png')

## Section 7: Firm-Level Evidence → Table 5, Fig 8

In [ ]:
# Table 5
pre_mean  = df[df['fiscal_year'].between(2020,2022)].groupby('ticker')['ln_revenue'].mean()
post_mean = df[df['fiscal_year'].between(2023,2025)].groupby('ticker')['ln_revenue'].mean()
growth = (post_mean - pre_mean).reset_index()
growth.columns = ['ticker','rev_growth']

sme = lit.merge(pre_rev, on='ticker')
sme = sme[sme['pre_rev_m']<200].merge(growth, on='ticker', how='left')
names = panel[['ticker','company_name']].drop_duplicates()
sme = sme.merge(names, on='ticker', how='left')

high = sme.nlargest(8,'lit_score')
low  = sme.nsmallest(8,'lit_score')

print('=== TABLE 5: REPRESENTATIVE FIRMS ===')
print(f'{"Group":<6} {"Ticker":<6} {"Company":<28} {"Score":<6} {"LLM":<5} {"ΔRev":<8}')
print('-'*62)
for grp, df_g in [("HIGH", high), ("LOW", low)]:
    for _, r in df_g.iterrows():
        d = (np.exp(r['rev_growth'])-1)*100
        print(f'{grp:<6} {r["ticker"]:<6} '
              f'{str(r["company_name"])[:26]:<28} '
              f'{r["lit_score"]:<6.0f} {r["early_ai"]:<5.0f} {d:+.1f}%')

print(f'\nHIGH mean ΔRev: {(np.exp(high["rev_growth"].mean())-1)*100:.1f}%')
print(f'LOW  mean ΔRev: {(np.exp(low["rev_growth"].mean())-1)*100:.1f}%')

In [ ]:
# Figure 8: Score vs Revenue scatter
scatter_df = sme.dropna(subset=['lit_score','rev_growth'])
r, p = stats.pearsonr(scatter_df['lit_score'], scatter_df['rev_growth'])

fig, ax = plt.subplots(figsize=(9, 6))
med_score = scatter_df['lit_score'].median()
colors = ['#e74c3c' if s>=med_score else '#2980b9' for s in scatter_df['lit_score']]
ax.scatter(scatter_df['lit_score'], scatter_df['rev_growth'],
    c=colors, alpha=0.7, s=60, edgecolors='white', linewidths=0.5)

z = np.polyfit(scatter_df['lit_score'], scatter_df['rev_growth'], 1)
x_line = np.linspace(scatter_df['lit_score'].min(), scatter_df['lit_score'].max(), 100)
ax.plot(x_line, np.polyval(z, x_line), 'k--', lw=2, alpha=0.6)
ax.axhline(0, color='black', lw=0.8)

for _, row in scatter_df.nlargest(3,'lit_score').iterrows():
    ax.annotate(row['ticker'], xy=(row['lit_score'], row['rev_growth']),
        xytext=(3,3), textcoords='offset points', fontsize=7, color='#e74c3c')
for _, row in scatter_df.nsmallest(3,'lit_score').iterrows():
    ax.annotate(row['ticker'], xy=(row['lit_score'], row['rev_growth']),
        xytext=(3,3), textcoords='offset points', fontsize=7, color='#2980b9')

ax.set_xlabel('Literature Replicability Score (1-100)', fontsize=11)
ax.set_ylabel('Post-shock Revenue Growth\n(ln(Rev) 2023-2025 minus 2020-2022)', fontsize=10)
ax.set_title(f'Figure 8: Score vs Post-Shock Revenue Growth\n'
    f'SME <$200M (n={len(scatter_df)}) | r={r:.3f}, p={p:.3f}',
    fontsize=11, fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', label=f'High (score≥{med_score:.0f})'),
    mpatches.Patch(color='#2980b9', label=f'Low (score<{med_score:.0f})'),
    plt.Line2D([0],[0], color='k', ls='--', label=f'OLS fit (r={r:.3f})')
], fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/figures/fig8_score_revenue_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: fig8_score_revenue_scatter.png | r={r:.3f}, p={p:.3f}')

## Section 8: Size Heterogeneity → Table 3, Fig 5

In [ ]:
print('=== TABLE 3: SIZE HETEROGENEITY ===')
t3 = pd.DataFrame({
    'Sample':  ['Full scored', '<$500M', '<$300M', '<$200M'],
    'N':       [116, 116, 106, 94],
    'Beta':    [-0.541, -0.541, -0.591, -0.604],
    'WCB_p':   [0.002, 0.002, 0.003, 0.003],
    'Sig':     ['***', '***', '***', '***']
})
print(t3.to_string(index=False))
print('→ Monotonically strengthens as firms get smaller ✅')
print('→ Consistent with switching cost theory ✅')

# Figure 5
fig, ax = plt.subplots(figsize=(9, 5))
cutoffs  = ['Full', '<$500M', '<$300M', '<$200M']
betas    = [-0.541, -0.541, -0.591, -0.604]
ses      = [0.202, 0.202, 0.210, 0.217]
ns       = [116, 116, 106, 94]
colors_h = ['#95a5a6','#f39c12','#e67e22','#e74c3c']

bars = ax.bar(range(4), [-b for b in betas],
    color=colors_h, alpha=0.85, width=0.5,
    yerr=ses, capsize=5, error_kw={'linewidth':1.5})
for i, (b, n, s) in enumerate(zip(betas, ns, ses)):
    ax.text(i, -b+s+0.01, f'β={b:.3f}***\nn={n}',
        ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(range(4))
ax.set_xticklabels(cutoffs, fontsize=11)
ax.set_ylabel('Effect Size (−β, log points)', fontsize=11)
ax.set_title('Figure 5: Size Heterogeneity — All p<0.005***',
    fontsize=11, fontweight='bold')
ax.grid(alpha=0.3, axis='y'); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig(f'{BASE}/figures/fig5_size_heterogeneity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig5_size_heterogeneity.png')

## Section 9: Scoring Pipeline & Sample Funnel → Fig 6, Fig 7

In [ ]:
# Figure 6: Scoring pipeline
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 7); ax.axis('off')
fig.suptitle('Figure 6: Literature-Grounded Replicability Scoring Pipeline',
    fontsize=12, fontweight='bold', y=0.98)

def box(ax, x, y, w, h, text, color, fontsize=9):
    rect = FancyBboxPatch((x-w/2, y-h/2), w, h,
        boxstyle='round,pad=0.1', facecolor=color,
        edgecolor='white', linewidth=1.5, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center',
        fontsize=fontsize, color='white', fontweight='bold',
        zorder=4, multialignment='center')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
        arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2), zorder=2)

for (x, text, color) in [
    (1.5, 'SEC EDGAR\n10-K Item 1\n"Business Description"', '#2980b9'),
    (4.0, 'Pre-shock text\n(~3,000 chars)\n2021-2022 filing', '#16a085'),
    (6.8, 'LLM-as-Judge\n(claude-opus-4-5)\nStructured rubric', '#8e44ad'),
    (10.0, 'Replicability\nScore\n1-100', '#e74c3c'),
    (12.5, 'Treatment\nVariable\nρᵢˡⁱᵗ / 100', '#c0392b'),
]:
    box(ax, x, 5.5, 2.5, 1.2, text, color, 8)

for x1, x2 in [(2.75, 2.9), (5.1, 5.55), (8.05, 8.9), (11.1, 11.4)]:
    arrow(ax, x1, 5.5, x2, 5.5)

box(ax, 6.8, 3.2, 5.5, 1.6,
    'Literature Rubric (Eloundou 2024 + Acemoglu 2022 + Brynjolfsson 2023)\n'
    '1-20: Pure E0 | 21-40: Mostly E0 | 41-60: Mixed\n'
    '61-80: Mostly E1 | 81-100: Pure E1',
    '#34495e', 8)
arrow(ax, 6.8, 4.9, 6.8, 4.0)

ax.text(7.0, 0.4, 'r = 0.895 with LLM Judge holistic | r = 0.299 with O*NET task-based',
    ha='center', fontsize=9, style='italic', color='#7f8c8d')

plt.tight_layout()
plt.savefig(f'{BASE}/figures/fig6_scoring_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig6_scoring_pipeline.png')

In [ ]:
# Figure 7: Sample funnel
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10); ax.set_ylim(0, 9); ax.axis('off')
fig.suptitle('Figure 7: Sample Construction Funnel',
    fontsize=12, fontweight='bold', y=0.98)

levels = [
    (7.5, 8.2, 'NYSE/Nasdaq listed firms', 7509, '#2980b9'),
    (6.5, 7.0, 'SIC 7370-7379 (Software)', 530, '#2471a3'),
    (5.5, 5.8, 'IPO before 2020Q1 + B2B', 173, '#1a6698'),
    (4.5, 4.6, 'Revenue data (full universe)', 143, '#155d86'),
    (3.5, 3.4, 'Lit Rubric scored (SME $500M)', 116, '#e67e22'),
    (3.0, 2.2, 'Primary sample (SME $200M)', 94, '#e74c3c'),
]

for width, y, label, n, color in levels:
    x_start = 5 - width/2
    rect = FancyBboxPatch((x_start, y-0.45), width, 0.9,
        boxstyle='round,pad=0.05', facecolor=color,
        edgecolor='white', linewidth=1.5, alpha=0.9, zorder=3)
    ax.add_patch(rect)
    ax.text(5, y, f'{label}  (n={n:,})',
        ha='center', va='center',
        fontsize=9, color='white', fontweight='bold', zorder=4)

for y1, y2 in [(7.75,7.45),(6.55,6.25),(5.35,5.05),(4.15,3.85),(2.95,2.65)]:
    ax.annotate('', xy=(5,y2), xytext=(5,y1),
        arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2))

plt.tight_layout()
plt.savefig(f'{BASE}/figures/fig7_sample_funnel.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig7_sample_funnel.png')

## Section 10: Three-Period Analysis → Table 6, Fig 9

In [ ]:
print('=== TABLE 6: THREE-PERIOD ANALYSIS ===')
t6 = pd.DataFrame({
    'Period':  ['Early AI (2022Q4-2023Q4)', 'Advanced AI (2024Q1+)'],
    'Beta_200M': [-0.451, -0.686],
    'p_200M':    [0.022, 0.008],
    'Sig_200M':  ['**', '***'],
    'Beta_500M': [-0.402, -0.615],
    'p_500M':    [0.018, 0.011],
    'Sig_500M':  ['**', '**'],
})
print(t6.to_string(index=False))
print(f'\nIntensification SME $200M: {0.686/0.451:.2f}x')
print(f'Intensification SME $500M: {0.615/0.402:.2f}x')

# Figure 9
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
fig.suptitle('Figure 9: Three-Period Analysis\n'
    'Early AI (2022Q4-2023Q4) vs Advanced AI (2024Q1+)',
    fontsize=11, fontweight='bold')

data = {
    'SME <$200M': {'Early AI': (-0.451,0.022), 'Advanced AI': (-0.686,0.008)},
    'SME <$500M': {'Early AI': (-0.402,0.018), 'Advanced AI': (-0.615,0.011)}
}
ses_d = {
    'SME <$200M': {'Early AI': 0.184, 'Advanced AI': 0.244},
    'SME <$500M': {'Early AI': 0.171, 'Advanced AI': 0.227}
}

for ax, (sample, vals) in zip(axes, data.items()):
    periods = list(vals.keys())
    betas  = [-vals[p][0] for p in periods]
    ps     = [vals[p][1]  for p in periods]
    errs   = [ses_d[sample][p] for p in periods]

    ax.bar(periods, betas, color=['#f39c12','#e74c3c'],
        alpha=0.85, width=0.45,
        yerr=errs, capsize=6,
        error_kw={'linewidth':1.5, 'ecolor':'#2c3e50'})

    for i, (b, p, se) in enumerate(zip(betas, ps, errs)):
        stars = '***' if p<0.01 else '**'
        ax.text(i, b+se+0.01, f'β={-b:.3f}{stars}\np={p:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

    ratio = betas[1]/betas[0]
    ax.text(0.5, max(betas)+max(errs)+0.15,
        f'{ratio:.2f}x more intense',
        ha='center', fontsize=9, color='#2c3e50', style='italic')

    n = 94 if '200' in sample else 116
    ax.set_title(f'{sample} (n={n})', fontsize=10)
    ax.set_ylabel('Effect Size (−β)', fontsize=10)
    ax.set_ylim(0, max(betas)+max(errs)+0.35)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{BASE}/figures/fig9_three_period.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig9_three_period.png')

## Section 11: Final Results Summary

In [ ]:
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('Generative AI as a Task Shock — TU Munich')
print('=' * 60)

print('\nPRIMARY RESULT:')
print('  Lit Rubric SME $200M: β=-0.604, p=0.003*** (n=94)')
print('  Lit Rubric SME $500M: β=-0.541, p=0.002*** (n=116)')

print('\nROBUSTNESS:')
print('  LLM Judge:   β=-0.426, p=0.031**  r=0.895')
print('  O*NET alpha: β=-0.335, p=0.059*   r=0.299')

print('\nIDENTIFICATION:')
print('  Placebo:     p=0.157 ✅')
print('  Pre-trends:  0/11 sig ✅')
print('  GM null:     p>0.4 ✅ (quantity channel)')
print('  Scatter:     r=-0.274, p=0.008 ✅')

print('\nSIZE HETEROGENEITY:')
print('  Full → $200M: -0.541 → -0.604 (monotonic) ✅')

print('\nTHREE-PERIOD:')
print('  Early AI:    β=-0.451, p=0.022**')
print('  Advanced AI: β=-0.686, p=0.008***')
print('  Intensification: 1.52x ✅')

print('\nFIRM-LEVEL:')
print('  HIGH group mean ΔRev: +1.8%')
print('  LOW  group mean ΔRev: +104.1%')

figs = [f for f in sorted(os.listdir(f'{BASE}/figures')) if f.endswith('.png')]
print(f'\nFigures generated: {len(figs)}')
for f in figs:
    print(f'  {f}')